# CineSuggest Movie Recommendation System
## Phase 3: Exploratory Data Analysis (EDA)

This notebook explores the preprocessed MovieLens dataset. It visualizes rating distributions, user activity patterns, genre popularity, temporal release trends, and correlations between rating volumes and average scores. These insights are critical for guiding recommendation model design.

In [ ]:
import sys
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR
from src.utils.helpers import read_csv, display_df_info

### Load the Processed Datasets

In [ ]:
movies_df = pd.read_csv(PROCESSED_DATA_DIR / 'processed_movies.csv')
ratings_df = pd.read_csv(PROCESSED_DATA_DIR / 'processed_ratings.csv')

print("Movies DataFrame Shape:", movies_df.shape)
print("Ratings DataFrame Shape:", ratings_df.shape)

### 1. Movie Rating Frequencies

Analyzing the rating distribution helps identify skewness and user rating tendencies.

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(x='rating', data=ratings_df, color='#ff007f', edgecolor='black', linewidth=0.7)
plt.title('Movie Rating Frequencies')
plt.xlabel('Rating Value')
plt.ylabel('Count')
plt.show()

### 2. User Activity Analysis

Understanding ratings count per user lets us identify power users and assess collaborative filtering density.

In [ ]:
user_counts = ratings_df['userId'].value_counts()
plt.figure(figsize=(9, 5))
sns.histplot(user_counts, kde=True, color='#7f00ff', bins=30, edgecolor='black', linewidth=0.7)
plt.title('Distribution of User Activity Levels')
plt.xlabel('Ratings per User')
plt.ylabel('Frequency')
plt.axvline(user_counts.median(), color='red', linestyle='--', label=f'Median: {int(user_counts.median())}')
plt.legend()
plt.show()

### 3. Movie Release Trends Over Time

Let's see how many movies are in the catalog across different release years.

In [ ]:
year_counts = movies_df['release_year'].dropna().astype(int).value_counts().sort_index()
plt.figure(figsize=(10, 5))
plt.plot(year_counts.index, year_counts.values, color='#ff007f', linewidth=2)
plt.fill_between(year_counts.index, year_counts.values, color='#ff007f', alpha=0.2)
plt.title('Movies Released per Year')
plt.xlabel('Year')
plt.ylabel('Count of Releases')
plt.show()

### 4. Correlation: Rating Count vs. Mean Rating

This scatter/joint plot shows whether popular movies tend to have higher average scores.

In [ ]:
rated_movies = movies_df[movies_df['rating_count'] > 0]
sns.jointplot(x='rating_count', y='rating_mean', data=rated_movies, kind='hex', color='#7f00ff', height=6)
plt.show()

## Summary of Findings

1. **Rating Distribution Bias**: The dataset is skewed towards positive reviews, with 4.0 being the most frequent rating score. This means recommendation systems need to normalize scores or handle user-specific baselines.
2. **Long-Tail Activity**: User interaction density is highly skewed. A very small fraction of users contributes to the majority of ratings. This requires regularization to prevent a few power users from dictating suggestions.
3. **Cold-Start Problem**: A massive fraction of the movie catalog has only 1 rating. Collaborating filtering models will fail on these items, emphasizing the importance of content-based meta similarity.